### Titanic Univariate Data Generation

This notebook loads the Titanic dataset, cleans it, and generates three datasets with MCAR, MAR, and MNAR univariate missingness in the `age` column using mdatagen.

Run this notebook to create the different missing mechanism versions of the titanic dataset in CSV files, that are used by `../analysis_uni.ipynb`.   
The CSVs are saved in the same `data/` folder as this notebook.

In [1]:
import numpy as np 
import seaborn as sns

from mdatagen.univariate.uMCAR import uMCAR
from mdatagen.univariate.uMAR import uMAR
from mdatagen.univariate.uMNAR import uMNAR

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import plotly.io as pio
pio.renderers.default = "notebook"

In [2]:
# intial data set loading
_df_raw = sns.load_dataset("titanic")

# check how many missing values we have in the dataset
_df_raw.isna().sum()

# check dataset shape
_df_raw.shape

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

(891, 15)

We can see that we have too many missing values in the "deck" column, so we will drop it for now

As for the missing values in the "age" and "embark_town" column, we will not impute but drop them.  
We do this because for our use case it is not relevant to keep those entries

In [3]:
_df_clean = _df_raw.drop(columns=["deck"])

_df_clean = _df_clean.dropna()

_df_clean.isna().sum()

_df_clean.shape

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

(712, 14)

Now we have a clean dataset to work with

The titanic dataset has categorical features and our library does not support categorical values, so we have two options: either we encode the features or delete them. For the purpose of the use case, it is safe to remove them.

In [4]:
# we will exclude the non-numeric columns for the analysis for simplicity
_df_numeric = _df_clean.select_dtypes(include=[np.number])

_df_numeric.isna().sum()

_df_numeric.shape

# set the seed to ensure reproducibility
np.random.seed(42)

survived    0
pclass      0
age         0
sibsp       0
parch       0
fare        0
dtype: int64

(712, 6)

On the next step, we define `X` and `y` because the library uses it to auto-select which feature to corrupt when `x_miss` is not specified. Since we always pass `x_miss='age'` directly, `survived` has no actual effect on the generated data. It is purely for the library to work correctly.

In [5]:
X = _df_numeric.copy().drop(columns=["survived"])
y = _df_clean["survived"].to_numpy()

We end up with less features but that is fine for this use case scenario. 

Now we will create different versions of the dataset with different missingness mechanisms.

### MCAR generation

Here we'll generate a dataset with MCAR missingness in the "age" column.  

Missingness in "age" is completely random and does not depend on any other variable

In [6]:
generator = uMCAR(X=X, y=y, missing_rate=20, x_miss='age')
df_mcar = generator.random()

df_mcar.isna().sum()

df_mcar.shape

pclass      0
age       142
sibsp       0
parch       0
fare        0
target      0
dtype: int64

(712, 6)

### MAR generation

Here we'll generate a dataset with MAR missingness in the "age" column.  

Missingness in "age" depends on the observed "pclass" column (passenger class: 1, 2, 3).   
The highest pclass values (third-class passengers) will receive missing age values.

In [7]:
generator = uMAR(X=X, y=y, missing_rate=20, x_miss='age', x_obs='pclass')
df_mar = generator.highest()

df_mar.isna().sum()

df_mar.shape

pclass      0
age       142
sibsp       0
parch       0
fare        0
target      0
dtype: int64

(712, 6)

### MNAR generation

Here we'll generate a dataset with MNAR missingness in the "age" column.  

Missingness in "age" depends on the unobserved values of "age" itself

In [8]:
generator = uMNAR(X=X, y=y, missing_rate=20, x_miss='age')
df_mnar = generator.run()

df_mnar.isna().sum()

df_mnar.shape

pclass      0
age       142
sibsp       0
parch       0
fare        0
target      0
dtype: int64

(712, 6)

Now we save the generated datasets to CSV so the analysis notebook can load them any time.

We drop the target column before saving since survived was only passed to satisfy the generator API and plays no role in the missingness analysis.

In [11]:
df_mcar.drop(columns=["target"]).to_csv("data/mcar_uni.csv", index=False)
df_mar.drop(columns=["target"]).to_csv("data/mar_uni.csv", index=False)
df_mnar.drop(columns=["target"]).to_csv("data/mnar_uni.csv", index=False)